In [ ]:
# Parameters
input_image = None  # papermill will inject the uploaded image path


In [ ]:
import torch
import torchvision.transforms as T
import numpy as np
from PIL import Image
from pathlib import Path
import segmentation_models_pytorch as smp
from PIL import Image as PILImage


In [ ]:
# Model configuration
IN_CH = 1
N_CLASSES = 6
ENCODER = 'resnet50'
MODEL_PATH = '/app/models/best_model.pth'  # adjust if different
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


In [ ]:
# Build model and load weights (handles common checkpoint formats)
model = smp.Unet(encoder_name=ENCODER, encoder_weights=None, in_channels=IN_CH, classes=N_CLASSES)
state = torch.load(MODEL_PATH, map_location='cpu')
if isinstance(state, dict):
    # common keys: 'model' or 'state_dict'
    if 'model' in state:
        state_dict = state['model']
    elif 'state_dict' in state:
        state_dict = state['state_dict']
    else:
        state_dict = state
else:
    state_dict = state

# remove possible 'module.' prefixes
clean_state = {}
for k, v in state_dict.items():
    new_k = k.replace('module.', '') if k.startswith('module.') else k
    clean_state[new_k] = v

model.load_state_dict(clean_state)
model.to(device)
model.eval()
print('Model loaded OK from', MODEL_PATH)


In [ ]:
assert input_image is not None, 'input_image parameter not provided to notebook.'
inp = Path(input_image)
img = Image.open(str(inp)).convert('L') if IN_CH==1 else Image.open(str(inp)).convert('RGB')
transform = T.Compose([T.Resize((512,512)), T.ToTensor()])
x = transform(img).unsqueeze(0).to(device)
with torch.no_grad():
    out = model(x)
    probs = torch.softmax(out, dim=1)
    pred_mask = torch.argmax(probs, dim=1).squeeze(0).cpu().numpy().astype('uint8')
out_dir = Path('notebooks/output_oct')
out_dir.mkdir(parents=True, exist_ok=True)
mask_path = out_dir / f'mask_{inp.stem}.png'
PILImage.fromarray(pred_mask).save(mask_path)
print('Saved mask to', mask_path)
mask_path
